# 용역 클러스터링

대상: 용역(servc) 7,513건 제목 벡터 (중복 제거본, BGE-M3 1024차원, float16 저장)

## 왜 용역만 따로 하는가

공사(cnstwk)는 클러스터링이 실패했다 - 제목이 "(가칭)탕정9초등학교 교사 신축 전기공사"처럼 대부분 고유명사라, 공사 성격이 아니라 지역명(탕정/에코/아라)으로 그룹이 갈렸다. 게다가 공사는 `main_cnstty_nm` 컬럼이 97.5% 채워져 있어 클러스터링이 필요 없었다.

용역은 상황이 다르다:

- 제목에 **업무 성격이 드러난다** ("2026년 정보시스템 통합 유지관리 용역", "장애인실태조사")
- 조달청 코드(`item_codes`)가 **16.3%밖에 없어** 대안이 부족하다
- 초기 탐색에서 K=8로 돌렸을 때 해석 가능한 그룹이 나왔다 (연구용역/IT구축/실태조사/건설감리/유지관리/정보보호/운영위탁/기능개선)

## 목표

LLM 프롬프트에 넣을 **카테고리 목록**을 만드는 것. 클러스터 자체를 태그로 쓰는 게 아니라, 클러스터가 알려주는 "축"을 참고해 사람이 목록을 확정한다.

## 0. 환경 확인

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import umap

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("sklearn", sklearn.__version__)

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

## 1. 데이터 로드

In [ ]:
from pathlib import Path

CACHE = Path("cache")

# float16으로 저장돼 있다(전송 크기 때문). sklearn 일부 함수가 float16에서
# 정밀도 경고를 내므로 float32로 올려서 쓴다 - 값 자체는 그대로다.
X = np.load(CACHE / "vectors_servc.npy").astype(np.float32)
df = pd.read_csv(CACHE / "meta_servc.csv")

print("벡터:", X.shape)
print("메타:", df.shape)
assert len(X) == len(df), "벡터와 메타 개수가 다르면 짝이 어긋난 것"
df.head()

## 2. 데이터 감 잡기

클러스터링 전에 어떤 제목들인지 눈으로 훑는다. 나중에 그룹 결과를 볼 때 기준이 된다.

In [ ]:
rng = np.random.default_rng(42)

df["title_len"] = df["title"].str.len()
print(df["title_len"].describe().round(1))

print("\n무작위 20건:")
for t in df["title"].sample(20, random_state=42):
    print(f"  {t[:70]}")

## 3. 그리드 탐색

차원축소와 클러스터링 파라미터는 서로 영향을 주므로 분리하지 않고 한 번에 탐색한다.
UMAP이 비싼 연산이라 바깥 루프에 두고, K는 안쪽에서 돌린다 - UMAP 1회 비용으로 K 여러 개를 본다.

**평가를 원본 1024차원에서 하는 이유**: UMAP은 이웃을 뭉치게 만드는 알고리즘이라 UMAP 공간에서 실루엣을 재면 "심하게 뭉치는 설정"이 무조건 이긴다. 원본 공간에서 재야 모든 UMAP 설정이 같은 잣대로 비교된다.

`min_dist=0.0` 고정 - 클러스터링 목적일 때의 표준값이다(0.1~0.5는 시각화용).

### 공사에서 얻은 결론을 여기 그대로 적용하지 않는다

공사 그리드에서는 `n_components`가 결과에 둔감했고 HDBSCAN이 노이즈 27%에 38개로 과분할됐다.
하지만 그건 **공사 데이터의 특성**(고유명사 지배)에서 나온 결과라, 성격이 다른 용역에 그대로
적용할 근거가 없다. 오히려 용역에는 HDBSCAN이 유리할 수도 있다:

- 카테고리가 몇 개인지 모르는 상태라 K를 미리 안 정해도 되는 게 장점
- 애매한 공고를 노이즈로 빼주므로 억지 분류를 피할 수 있음

그래서 공사와 **같은 범위로 다시 탐색**하고, 용역에서는 어떻게 나오는지 직접 확인한다.
HDBSCAN은 노이즈 비율도 함께 기록해 공사(27%)와 비교한다.

In [ ]:
import time
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize

Xn = normalize(X)  # BGE-M3는 이미 정규화돼 있지만, 모델이 바뀌어도 안전하게

N_COMPONENTS = [5, 10, 15, 20, 30, 50, 100]
N_NEIGHBORS = [5, 15, 50]
K_GRID = list(range(4, 21))
HDBSCAN_GRID = [(mcs, ms) for mcs in (15, 30, 60) for ms in (5, 15)]


def evaluate(labels):
    """원본 공간 실루엣. 노이즈(-1)는 제외하고 잰다. 평가 불가면 None."""
    mask = labels >= 0
    if mask.sum() < 100 or len(set(labels[mask])) < 2:
        return None
    return silhouette_score(Xn[mask], labels[mask], metric="cosine",
                            sample_size=min(2000, int(mask.sum())), random_state=42)


rows = []
t0 = time.perf_counter()
done = 0
total = len(N_COMPONENTS) * len(N_NEIGHBORS)

for n_comp in N_COMPONENTS:
    for n_nb in N_NEIGHBORS:
        emb = umap.UMAP(n_components=n_comp, n_neighbors=n_nb, min_dist=0.0,
                        metric="cosine", random_state=42).fit_transform(Xn)

        for k in K_GRID:
            labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(emb)
            rows.append({"model": "kmeans", "n_components": n_comp, "n_neighbors": n_nb,
                         "param": f"K={k}", "n_clusters": k, "noise_ratio": 0.0,
                         "silhouette": evaluate(labels)})

        for mcs, ms in HDBSCAN_GRID:
            labels = HDBSCAN(min_cluster_size=mcs, min_samples=ms).fit_predict(emb)
            n_cl = len(set(labels)) - (1 if -1 in labels else 0)
            rows.append({"model": "hdbscan", "n_components": n_comp, "n_neighbors": n_nb,
                         "param": f"mcs={mcs},ms={ms}", "n_clusters": n_cl,
                         "noise_ratio": round(float((labels == -1).mean()), 3),
                         "silhouette": evaluate(labels)})

        done += 1
        print(f"  UMAP {done}/{total} (n_comp={n_comp}, n_nb={n_nb}) "
              f"누적 {time.perf_counter()-t0:.0f}초", flush=True)

results = pd.DataFrame(rows)
results.to_csv(CACHE / "grid_servc.csv", index=False, encoding="utf-8-sig")
print(f"\n조합 {len(results)}개 완료")

### 3-1. 상위 조합

In [ ]:
# HDBSCAN은 노이즈가 과하면 실루엣이 높아도 실용성이 없으므로 30% 초과는 걸러낸다
valid = results[results["silhouette"].notna() & (results["noise_ratio"] <= 0.3)]

print(f"전체 {len(results)}개 중 유효 {len(valid)}개")
print("\n모델별 최고 실루엣:")
print(valid.groupby("model")["silhouette"].max().round(4))
print("\nHDBSCAN 노이즈 비율 분포 (공사는 평균 24%였음):")
hd = results[results["model"] == "hdbscan"]
print(hd["noise_ratio"].describe().round(3))

valid.sort_values("silhouette", ascending=False).head(15)

### 3-2. 파라미터별 경향

In [ ]:
km = valid[valid["model"] == "kmeans"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# KMeans K별 - n_components가 실제로 영향을 주는지 (공사에서는 둔감했음)
for n_comp, g in km.groupby("n_components"):
    axes[0].plot(g.groupby("n_clusters")["silhouette"].max(), marker="o", label=f"{n_comp}")
axes[0].set_xlabel("K"); axes[0].set_ylabel("최고 실루엣")
axes[0].set_title("K별 (n_components 비교)"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

for n_nb, g in km.groupby("n_neighbors"):
    axes[1].plot(g.groupby("n_clusters")["silhouette"].max(), marker="o", label=f"{n_nb}")
axes[1].set_xlabel("K"); axes[1].set_title("K별 (n_neighbors 비교)")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

# HDBSCAN - 노이즈 비율과 실루엣의 관계 (노이즈를 늘려서 점수를 버는지 확인)
hd_valid = valid[valid["model"] == "hdbscan"]
axes[2].scatter(hd_valid["noise_ratio"], hd_valid["silhouette"], alpha=0.5, s=20)
axes[2].set_xlabel("노이즈 비율"); axes[2].set_ylabel("실루엣")
axes[2].set_title("HDBSCAN: 노이즈 vs 점수")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. 그룹 내용 확인

**여기가 실제 판단 지점이다.** 실루엣 점수는 참고일 뿐이고, 그룹이 카테고리로 말이 되는지는 제목을 직접 읽어야 안다.

공사 클러스터링에서 중심 근처 표본만 보고 "잘 나뉘었다"고 판단했다가, 실제로는 지역명으로 갈린 걸 놓칠 뻔했다. 중심에 가까운 건 그 그룹의 가장 전형적인 것들이라 당연히 일관돼 보인다. 그래서 여기서는 **세 가지 표본을 함께** 본다:

- **중심 근처** - 그룹의 전형
- **무작위** - 편향 없는 실제 분포
- **경계(중심에서 먼 것)** - 이 그룹이 진짜 하나의 카테고리인지 판정하는 핵심

In [ ]:
def inspect(n_comp, n_nb, model="kmeans", k=None, mcs=None, ms=None, n_show=5):
    """지정한 조합으로 다시 클러스터링하고 그룹별 표본 3종을 출력한다.

    random_state가 고정돼 있어 그리드 탐색 때와 같은 결과가 재현된다.

    사용 예:
        inspect(10, 15, "kmeans", k=12)
        inspect(10, 15, "hdbscan", mcs=30, ms=5)
    """
    emb = umap.UMAP(n_components=n_comp, n_neighbors=n_nb, min_dist=0.0,
                    metric="cosine", random_state=42).fit_transform(Xn)

    if model == "kmeans":
        labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(emb)
    else:
        labels = HDBSCAN(min_cluster_size=mcs, min_samples=ms).fit_predict(emb)
        print(f"그룹 {len(set(labels)) - (1 if -1 in labels else 0)}개, "
              f"노이즈 {(labels == -1).mean()*100:.1f}%")

    for c in sorted(set(labels)):
        idx = np.where(labels == c)[0]
        name = "노이즈(미분류)" if c == -1 else f"그룹 {c}"

        center = emb[idx].mean(axis=0)
        order = np.argsort(np.linalg.norm(emb[idx] - center, axis=1))

        print(f"\n{'='*66}")
        print(f"{name} - {len(idx)}건")
        print("-- 중심 근처(전형) --")
        for i in idx[order][:n_show]:
            print(f"   {df['title'][i][:64]}")
        print("-- 무작위 --")
        for i in rng.choice(idx, min(n_show, len(idx)), replace=False):
            print(f"   {df['title'][i][:64]}")
        print("-- 경계(중심에서 먼 것) --")
        for i in idx[order][-n_show:]:
            print(f"   {df['title'][i][:64]}")

    return labels

### 4-1. 최고 조합 확인

3-1의 상위 조합 값을 넣어서 실행한다.

In [ ]:
best = valid.sort_values("silhouette", ascending=False).iloc[0]
print(f"최고 조합: {best.model} / n_comp={int(best.n_components)} / "
      f"n_nb={int(best.n_neighbors)} / {best.param} / 실루엣={best.silhouette:.4f}\n")

if best.model == "kmeans":
    labels_best = inspect(int(best.n_components), int(best.n_neighbors),
                          "kmeans", k=int(best.n_clusters))
else:
    mcs, ms = [int(v.split("=")[1]) for v in best.param.split(",")]
    labels_best = inspect(int(best.n_components), int(best.n_neighbors),
                          "hdbscan", mcs=mcs, ms=ms)

### 4-2. 다른 조합과 비교

실루엣이 최고인 조합이 카테고리로도 최적이라는 보장은 없다. 공사에서 실제로 그랬다 -
HDBSCAN이 실루엣 1등이었지만 지역명(탕정/에코/아라)으로 잘게 쪼갠 결과였다.

K를 바꿔가며, 그리고 KMeans와 HDBSCAN을 번갈아 보면서 **어느 쪽이 이름 붙이기 좋은지**
직접 비교한다.

In [ ]:
# 값을 바꿔가며 반복 실행한다
labels_alt = inspect(10, 15, "kmeans", k=10)

# HDBSCAN도 비교해볼 것:
# labels_alt = inspect(10, 15, "hdbscan", mcs=30, ms=5)

## 5. 결론

실행 후 채운다.

- 채택한 조합 (모델 / n_components / n_neighbors / 파라미터 / 실루엣):
- 그룹이 카테고리로 말이 되는가 (**경계 표본까지** 봤을 때):
- 공사처럼 고유명사로 갈린 그룹이 있는가:
- KMeans vs HDBSCAN 어느 쪽이 나았는가 / HDBSCAN 노이즈는 몇 %였는가 (공사는 27%):
- `n_components`가 실제로 영향을 줬는가 (공사에서는 둔감했음):
- **카테고리 목록 초안**:
    1. 
    2. 
    3. 
- 클러스터링으로 안 잡히는 성격이 있는가 (코드 그룹과 대조 필요):

### 다음 단계

- 데스크탑에서 조달청 코드(`item_codes`) 그룹과 대조 - 클러스터와 코드가 얼마나 일치하는지(ARI)
- 목록 확정 후 `pipeline/realtime/src/extractors/llm/` 프롬프트에 반영